# Fase 03b: Recuperación de Cuarentena
**Objetivo:** Reparar automáticamente registros en cuarentena imputando valores estadísticos (medianas) de Silver, y devolverlos al pipeline principal.
**Entradas:** Parquet desde `medicalproyect-processed/quarantine/` + `medicalproyect-processed/silver/`.
**Salidas:** Reparados en `medicalproyect-processed/silver/` (append), no recuperables en `medicalproyect-processed/rejected/` + logs.


## 1. Inicialización y Configuración


In [1]:
def _subir_log(sufijo, storage_account):
    try:
        from notebookutils import mssparkutils
        import os
        from datetime import datetime
        fecha = datetime.now().strftime('%Y%m%d_%H%M%S')
        ruta_local = "file://" + os.path.abspath(log_filename)
        ruta_destino = f"abfss://medicalproyect-logs@{storage_account}.dfs.core.windows.net/{sufijo}.log"
        mssparkutils.fs.cp(ruta_local, ruta_destino)
    except Exception:
        pass


# ── Función helper para notificaciones por Telegram ──
def _enviar_telegram(mensaje):
    try:
        from notebookutils import mssparkutils
        import requests
        token = mssparkutils.credentials.getSecret(NOMBRE_BOVEDA, "TelegramToken", NOMBRE_PUENTE)
        chat_id = mssparkutils.credentials.getSecret(NOMBRE_BOVEDA, "TelegramChatId", NOMBRE_PUENTE)
        url = f"https://api.telegram.org/bot{token}/sendMessage"
        requests.post(url, json={"chat_id": chat_id, "text": mensaje, "parse_mode": "HTML"}, timeout=10)
    except Exception:
        logger.warning(f"Telegram falló: {e}")
        pass

# ── Función helper: sube el log al Data Lake ──
        from notebookutils import mssparkutils
        import os
        from datetime import datetime
        fecha = datetime.now().strftime('%Y%m%d_%H%M%S')
        ruta_local = "file://" + os.path.abspath(log_filename)
        ruta_destino = f"abfss://medicalproyect-logs@{storage_account}.dfs.core.windows.net/{sufijo}.log"
        mssparkutils.fs.cp(ruta_local, ruta_destino)
    except Exception:
        pass

# ══════════════════════════════════════════════════════════════
# 01c — RECUPERACIÓN DE CUARENTENA
# Entrada:  medicalproyect-raw/quarantine/
# Salida:   registros reparados → medicalproyect-processed/silver/  (append)
#           no recuperables     → medicalproyect-raw/rejected/
# ══════════════════════════════════════════════════════════════

# Configuramos el logging para registrar todo el proceso en archivo y consola
import os
import logging
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, size, split, when, median, avg
)

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

log_filename = "pipeline_recuperacion.log"
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-7s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[
        logging.FileHandler(log_filename, mode='a', encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("Medical.Recuperacion")

logger.info("=" * 60)
logger.info("INICIO PIPELINE DE RECUPERACIÓN DE CUARENTENA")
logger.info("=" * 60)

# Definimos las rutas de los contenedores en Azure Data Lake
STORAGE_ACCOUNT   = "stproyectomastergrupo3"
CONTAINER_RAW     = "medicalproyect-raw"

# ── Configuración de Key Vault ──
NOMBRE_BOVEDA = "kv-medicalproyect"
NOMBRE_PUENTE = "AzureKeyVault"

CONTAINER_SILVER  = "medicalproyect-processed"
CONTAINER_LOGS    = "medicalproyect-logs"

base_quarantine = f"abfss://{CONTAINER_SILVER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/quarantine"
base_rejected   = f"abfss://{CONTAINER_SILVER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/rejected"
base_silver     = f"abfss://{CONTAINER_SILVER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/silver"

# Arrancamos la sesion de Spark para procesar los datos en el cluster
spark = SparkSession.builder.appName("Medical_Recuperacion").getOrCreate()
logger.info("SparkSession creada.")
logger.info("📱 Enviando notificación de inicio...")
_enviar_telegram("🚀 INICIADO: 03B Recuperacion Cuarentena")


## 2. Función Helper: Recuperación de Cuarentena


In [2]:
# ══════════════════════════════════════════════════════════════
# HELPER PRINCIPAL
# ══════════════════════════════════════════════════════════════

# Funcion que decide si un registro en cuarentena se puede reparar o debe ir a rejected
def intentar_recuperar(df, col_motivo, reparaciones, nombre_tabla):
    """
    - reparaciones: dict {motivo_str: funcion(df) -> df}
    - Registros con 1 motivo reparable  → se reparan y van al Silver
    - Registros con >1 motivo o motivo no reparable → van a rejected
    """
    total_quar = df.count()
    logger.info(f"  [{nombre_tabla}] registros en quarantine: {total_quar:,}")

    if total_quar == 0:
        logger.info(f"  [{nombre_tabla}] sin registros que procesar.")
        return None, None

    # Contar motivos por registro (separados por |)
    df = df.withColumn(
        "n_motivos",
        size(split(col(col_motivo), r"\|"))
    )

    motivos_reparables = list(reparaciones.keys())

    # Recuperables: exactamente 1 motivo Y ese motivo tiene reparación definida
    df_recuperables = df.filter(
        (col("n_motivos") == 1) &
        col(col_motivo).isin(motivos_reparables)
    )

    # No recuperables: múltiples motivos o motivo sin reparación
    df_no_recuperables = df.filter(
        (col("n_motivos") > 1) |
        (~col(col_motivo).isin(motivos_reparables))
    )

    n_recuperables    = df_recuperables.count()
    n_no_recuperables = df_no_recuperables.count()
    logger.info(f"  [{nombre_tabla}] recuperables: {n_recuperables:,} | "
                f"no recuperables: {n_no_recuperables:,}")

    # Aplicar reparación a cada grupo por motivo
    df_reparado = None
    for motivo, fn_reparar in reparaciones.items():
        subset = df_recuperables.filter(col(col_motivo) == motivo)
        n = subset.count()
        if n > 0:
            subset_reparado = fn_reparar(subset)
            logger.info(f"    -> '{motivo}': {n:,} registros reparados.")
            df_reparado = subset_reparado if df_reparado is None \
                          else df_reparado.unionByName(subset_reparado)

    # Limpiar columnas de auditoría antes de mandar al Silver
    cols_auditoria = [col_motivo, "tabla_origen", "ts_alerta", "n_motivos"]
    if df_reparado is not None:
        cols_a_drop = [c for c in cols_auditoria if c in df_reparado.columns]
        df_reparado = df_reparado.drop(*cols_a_drop)

    # Guardar no recuperables en rejected
    if n_no_recuperables > 0:
        df_no_recuperables \
            .withColumn("motivo_rechazo", col(col_motivo)) \
            .withColumn("origen", lit("quarantine_no_recuperable")) \
            .write.mode("overwrite") \
            .parquet(f"{base_rejected}/{nombre_tabla}")
        logger.info(f"  [{nombre_tabla}] {n_no_recuperables:,} registros "
                    f"enviados a rejected.")

    # Devolvemos el DataFrame reparado y un resumen con las estadisticas
    return df_reparado, {
        "tabla":            nombre_tabla,
        "total_quarantine": total_quar,
        "recuperados":      n_recuperables,
        "rechazados":       n_no_recuperables
    }

# Lista donde acumulamos el resumen de cada tabla que procesamos
resumenes_rec = []
# Subida parcial de log tras esta seccion
_subir_log("calidad/recuperacion_recuperación_por_tabla", STORAGE_ACCOUNT)


## 3. Recuperación por Tabla


In [3]:
# ══════════════════════════════════════════════════════════════
# PATIENTS
# ══════════════════════════════════════════════════════════════
logger.info("")
logger.info("── Recuperando PATIENTS ──")

# Intentamos leer los datos de pacientes en cuarentena, si la carpeta no existe continuamos
try:
    df_quar_patients = spark.read.parquet(f"{base_quarantine}/patients")
except Exception:
    df_quar_patients = None
    logger.info("  [patients] carpeta quarantine no existe, nada que procesar.")

if df_quar_patients:

    # Calcular medianas desde Silver (datos limpios ya procesados)
    df_silver_patients = spark.read.parquet(f"{base_silver}/patients")
    medianas_patients  = df_silver_patients.select(
        avg("bmi").alias("med_bmi"),
        avg("systolic_bp").alias("med_systolic"),
        avg("diastolic_bp").alias("med_diastolic"),
        avg("heart_rate").alias("med_heart_rate"),
        avg("temperature_f").alias("med_temp_f")
    ).collect()[0]

    med_bmi        = medianas_patients["med_bmi"]
    med_systolic   = medianas_patients["med_systolic"]
    med_diastolic  = medianas_patients["med_diastolic"]
    med_heart_rate = medianas_patients["med_heart_rate"]
    med_temp_f     = medianas_patients["med_temp_f"]

    logger.info(f"  Medianas Silver — bmi:{med_bmi:.1f} | "
                f"systolic:{med_systolic:.1f} | diastolic:{med_diastolic:.1f} | "
                f"hr:{med_heart_rate:.1f} | temp_f:{med_temp_f:.1f}")

    reparaciones_patients = {
        "bmi_fuera_rango":
            lambda df: df.withColumn("bmi", lit(med_bmi)),

        "systolic_bp_extrema":
            lambda df: df.withColumn("systolic_bp", lit(round(med_systolic)).cast("integer")),

        "diastolic_bp_extrema":
            lambda df: df.withColumn("diastolic_bp", lit(round(med_diastolic)).cast("integer")),

        "heart_rate_extrema":
            lambda df: df.withColumn("heart_rate", lit(round(med_heart_rate)).cast("integer")),

        "temperatura_extrema":
            lambda df: df.withColumn("temperature_f", lit(med_temp_f)),

        "charlson_negativo":
            lambda df: df.withColumn("charlson_index", lit(0)),
    }

    # Ejecutamos la recuperacion: repara los registros posibles y guarda en Silver
    df_patients_rec, r = intentar_recuperar(
        df_quar_patients, "motivo_alerta", reparaciones_patients, "patients"
    )
    if df_patients_rec is not None and df_patients_rec.count() > 0:
        df_patients_rec.write.mode("append").parquet(f"{base_silver}/patients")
        logger.info(f"  [patients] reparados escritos en Silver.")
    if r:
        resumenes_rec.append(r)

In [4]:
# Inspeccionamos los datos de outcomes en cuarentena para ver que motivos de alerta hay
df_quar_outcomes = spark.read.parquet(f"{base_quarantine}/outcomes")
logger.info("Valores distintos en motivo_alerta:")
df_quar_outcomes.groupBy("motivo_alerta").count().orderBy("count", ascending=False).show(truncate=False)
logger.info("n_motivos:")
df_quar_outcomes.withColumn("n_motivos", size(split(col("motivo_alerta"), r"\|"))).groupBy("n_motivos").count().show()

In [5]:
# ══════════════════════════════════════════════════════════════
# OUTCOMES
# ══════════════════════════════════════════════════════════════
logger.info("")
logger.info("── Recuperando OUTCOMES ──")

# Intentamos leer los datos de outcomes en cuarentena
try:
    df_quar_outcomes = spark.read.parquet(f"{base_quarantine}/outcomes")
except Exception:
    df_quar_outcomes = None
    logger.info("  [outcomes] carpeta quarantine no existe, nada que procesar.")

if df_quar_outcomes:

    # Calculamos medianas de cargos y estancia desde Silver para imputar valores
    df_silver_outcomes = spark.read.parquet(f"{base_silver}/outcomes")
    med_charges = df_silver_outcomes.select(
        avg("total_charges_usd").alias("med_charges"),
        avg("length_of_stay_days").alias("med_los")
    ).collect()[0]

    med_charges_val = med_charges["med_charges"]
    med_los_val     = med_charges["med_los"]

    logger.info(f"  Medianas Silver — charges:{med_charges_val:.2f} | "
                f"los:{med_los_val:.1f}")

    reparaciones_outcomes = {
        # Cargos negativos → imputar con mediana
        "cargos_negativos":
            lambda df: df.withColumn("total_charges_usd", lit(med_charges_val)),

        # Estancia > 365 → imputar con mediana (casos extremos pero posibles)
        "estancia_superior_1anio":
            lambda df: df.withColumn("length_of_stay_days",
                                     lit(round(med_los_val)).cast("integer")),

        # icu_days > length_of_stay → recortar icu_days al valor de estancia
        "icu_days_mayor_estancia":
            lambda df: df.withColumn("icu_days", col("length_of_stay_days")),
    }

    # Ejecutamos la recuperacion y guardamos los registros reparados en Silver
    df_outcomes_rec, r = intentar_recuperar(
        df_quar_outcomes, "motivo_alerta", reparaciones_outcomes, "outcomes"
    )
    if df_outcomes_rec is not None and df_outcomes_rec.count() > 0:
        # Asegurar el mismo esquema que el notebook Silver original para TODAS las booleanas
        from pyspark.sql.functions import col
        df_outcomes_rec = df_outcomes_rec \
            .withColumn("icu_admission", col("icu_admission").cast("boolean")) \
            .withColumn("in_hospital_death", col("in_hospital_death").cast("boolean")) \
            .withColumn("readmitted_30d", col("readmitted_30d").cast("boolean"))
        
        df_outcomes_rec.write.mode("append").parquet(f"{base_silver}/outcomes")
    if r:
        resumenes_rec.append(r)

In [6]:
# ══════════════════════════════════════════════════════════════
# LAB RESULTS
# ══════════════════════════════════════════════════════════════
logger.info("")
logger.info("── Recuperando LAB_RESULTS ──")

# Intentamos leer los resultados de laboratorio en cuarentena
try:
    df_quar_lab = spark.read.parquet(f"{base_quarantine}/lab_results")
except Exception:
    df_quar_lab = None
    logger.info("  [lab_results] carpeta quarantine no existe, nada que procesar.")

if df_quar_lab:

    # Medianas por test_name desde Silver (cada analítica tiene su propia escala)
    df_silver_lab = spark.read.parquet(f"{base_silver}/lab_results")
    medianas_lab  = df_silver_lab.groupBy("test_name") \
                                 .agg(avg("value").alias("mediana_value")) \
                                 .collect()
    dict_medianas_lab = {row["test_name"]: row["mediana_value"] for row in medianas_lab}

    def reparar_valor_negativo(df):
        """Sustituye value negativo por la mediana de ese test_name."""
        df_out = df
        for test, mediana in dict_medianas_lab.items():
            df_out = df_out.withColumn(
                "value",
                when((col("test_name") == test) & (col("value") < 0), lit(mediana))
                .otherwise(col("value"))
            )
        return df_out

    def reparar_is_abnormal(df):
        """Recalcula is_abnormal comparando value con reference_low/high."""
        return df.withColumn(
            "is_abnormal",
            when(
                (col("value") < col("reference_low")) |
                (col("value") > col("reference_high")),
                lit(True)
            ).otherwise(lit(False))
        )

    # Definimos las reparaciones disponibles para laboratorio
    reparaciones_lab = {
        "valor_analitica_negativo":  reparar_valor_negativo,
        "is_abnormal_inconsistente": reparar_is_abnormal,
    }

    # Ejecutamos la recuperacion y guardamos los datos reparados en Silver
    df_lab_rec, r = intentar_recuperar(
        df_quar_lab, "motivo_alerta", reparaciones_lab, "lab_results"
    )
    if df_lab_rec is not None and df_lab_rec.count() > 0:
        df_lab_rec.write.mode("append").parquet(f"{base_silver}/lab_results")
        logger.info(f"  [lab_results] reparados escritos en Silver.")
    if r:
        resumenes_rec.append(r)

In [7]:
# ══════════════════════════════════════════════════════════════
# MEDICATIONS
# ══════════════════════════════════════════════════════════════
logger.info("")
logger.info("── Recuperando MEDICATIONS ──")

# Intentamos leer las medicaciones en cuarentena
try:
    df_quar_med = spark.read.parquet(f"{base_quarantine}/medications")
except Exception:
    df_quar_med = None
    logger.info("  [medications] carpeta quarantine no existe, nada que procesar.")

if df_quar_med:

    # Calculamos valores de referencia desde Silver para imputar datos erroneos
    df_silver_med  = spark.read.parquet(f"{base_silver}/medications")

    # Mediana de adherencia global
    med_adherence = df_silver_med.select(
        avg("adherence_pct").alias("med_adh")
    ).collect()[0]["med_adh"]

    # Medianas de duración por medicamento
    medianas_dur  = df_silver_med.groupBy("medication") \
                                 .agg(avg("duration_days").alias("med_dur")) \
                                 .collect()
    dict_dur = {row["medication"]: row["med_dur"] for row in medianas_dur}

    # Funcion para corregir duraciones negativas imputando la mediana de cada medicamento
    def reparar_duracion(df):
        df_out = df
        for med, mediana in dict_dur.items():
            df_out = df_out.withColumn(
                "duration_days",
                when(
                    (col("medication") == med) & (col("duration_days") < 0),
                    lit(round(mediana)).cast("integer")
                ).otherwise(col("duration_days"))
            )
        return df_out

    # Definimos las reparaciones para errores en medicaciones
    reparaciones_med = {
        # Adherencia fuera de rango → imputar con mediana global
        "adherencia_fuera_rango":
            lambda df: df.withColumn("adherence_pct", lit(med_adherence)),

        # Duración negativa → imputar con mediana por medicamento
        "duracion_negativa": reparar_duracion,

        # Fecha futura → no reparable, irá a rejected automáticamente
        # (no se incluye en reparaciones, el helper la manda a rejected)
    }

    # Recuperamos los registros reparables y los guardamos en Silver
    df_med_rec, r = intentar_recuperar(
        df_quar_med, "motivo_alerta", reparaciones_med, "medications"
    )
    if df_med_rec is not None and df_med_rec.count() > 0:
        df_med_rec.write.mode("append").parquet(f"{base_silver}/medications")
        logger.info(f"  [medications] reparados escritos en Silver.")
    if r:
        resumenes_rec.append(r)


    # ── Limpiar quarantine una vez procesado ──
    try:
        from notebookutils import mssparkutils
        mssparkutils.fs.rm(f"{base_quarantine}/medications", recurse=True)
        logger.info(f"  [medications] carpeta quarantine eliminada.")
    except Exception as e:
        logger.warning(f"  [patients] no se pudo eliminar quarantine: {e}")
# Subida parcial de log tras esta seccion
_subir_log("calidad/recuperacion_resumen_final", STORAGE_ACCOUNT)


## 4. Resumen Final y Cierre


In [8]:
# ══════════════════════════════════════════════════════════════
# RESUMEN FINAL Y LOGS
# ══════════════════════════════════════════════════════════════
# Mostramos el resumen de recuperacion en el log y en pantalla
logger.info("")
logger.info("=" * 60)
logger.info("RESUMEN FINAL DE RECUPERACIÓN")
logger.info("=" * 60)
logger.info(f"{'Tabla':<20} {'En quarantine':>14} {'Recuperados':>12} {'Rechazados':>12}")
logger.info("-" * 60)
for r in resumenes_rec:
    logger.info(
        f"{r['tabla']:<20} {r['total_quarantine']:>14,} "
        f"{r['recuperados']:>12,} {r['rechazados']:>12,}"
    )

print()
print("=" * 60)
logger.info("RESUMEN DE RECUPERACIÓN DE CUARENTENA")
print("=" * 60)
print(f"\n{'Tabla':<20} {'En quarantine':>14} {'Recuperados':>12} {'Rechazados':>12}")
print("-" * 60)
for r in resumenes_rec:
    print(
        f"{r['tabla']:<20} {r['total_quarantine']:>14,} "
        f"{r['recuperados']:>12,} {r['rechazados']:>12,}"
    )

# Calculamos los totales globales de todas las tablas procesadas
total_quar_all = sum(r['total_quarantine'] for r in resumenes_rec)
total_rec_all  = sum(r['recuperados']      for r in resumenes_rec)
total_rech_all = sum(r['rechazados']       for r in resumenes_rec)
pct_rec        = round(total_rec_all / total_quar_all * 100, 2) if total_quar_all > 0 else 0

print("-" * 60)
print(f"{'TOTAL':<20} {total_quar_all:>14,} {total_rec_all:>12,} {total_rech_all:>12,}")
print(f"\n  Tasa de recuperación: {pct_rec:.2f}%")
print("=" * 60)

# ── Cerrar logger y guardar log en Data Lake ──
for handler in logging.root.handlers[:]:
    handler.flush()
    handler.close()
    logging.root.removeHandler(handler)

try:
    from notebookutils import mssparkutils
    fecha_str  = datetime.now().strftime('%Y%m%d_%H%M')
    ruta_log   = f"abfss://{CONTAINER_LOGS}@{STORAGE_ACCOUNT}.dfs.core.windows.net/calidad/recuperacion_{fecha_str}.log"
    ruta_local = "file://" + os.path.abspath(log_filename)
    mssparkutils.fs.cp(ruta_local, ruta_log)
    print(f"\nLog guardado en: {ruta_log}")
except Exception as e:
    logger.warning(f"No se pudo copiar el log: {e}")

# Cerramos la sesion de Spark
logger.info("EXECUTION STATUS: SUCCESS")
_enviar_telegram("✅ FINALIZADO: 03B Recuperacion Cuarentena")
spark.stop()